# 03 - Resume Corpus RAG Q&A

This notebook retrieves resume chunks from Chroma and asks Gemini 3.5 Flash-Lite to answer questions using only the retrieved resume context. It works for one resume or multiple PDFs in `data/`.

In [6]:
from pathlib import Path
import os
from textwrap import dedent

import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if (current / "chroma_db").exists():
        return current
    if (current.parent / "chroma_db").exists():
        return current.parent
    raise FileNotFoundError("Run 02_vectorize_resume.ipynb first to create chroma_db.")


PROJECT_ROOT = find_project_root()
CHROMA_DIR = PROJECT_ROOT / "chroma_db"

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise EnvironmentError("GOOGLE_API_KEY is missing. Add it to your .env file.")

GENERATION_MODEL = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSIONS = 768
COLLECTION_NAME = "resume_rag"

genai_client = genai.Client(api_key=api_key)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION_NAME)

print(f"Loaded Chroma collection '{COLLECTION_NAME}' with {collection.count()} chunk(s).")

Loaded Chroma collection 'resume_rag' with 6 chunk(s).


In [7]:
def embed_query(query: str) -> list[float]:
    result = genai_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=EMBEDDING_DIMENSIONS,
        ),
    )
    return result.embeddings[0].values


def retrieve(query: str, k: int = 4) -> list[dict]:
    query_embedding = embed_query(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    matches = []
    for text, metadata, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        matches.append({"text": text, "metadata": metadata, "distance": distance})
    return matches


def format_context(matches: list[dict]) -> str:
    context_blocks = []
    for index, match in enumerate(matches, start=1):
        metadata = match["metadata"]
        context_blocks.append(
            f"[Context {index} | source={metadata['source']} | page={metadata['page']} | chunk={metadata['chunk_index']}]\n"
            f"{match['text']}"
        )
    return "\n\n".join(context_blocks)


def answer_question(question: str, k: int = 4, show_context: bool = False) -> str:
    matches = retrieve(question, k=k)
    context = format_context(matches)

    prompt = dedent(
        f"""
        Answer the user's question using only the resume context below.

        Rules:
        - If the answer is not supported by the context, say: "I don't know the answer based on the Data."
        - Be specific and concise.
        - When useful, mention the source filename and page number from the context.
        - Do not invent dates, employers, projects, skills, education, or experience.

        Resume context:
        {context}

        Question:
        {question}
        """
    ).strip()

    response = genai_client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(max_output_tokens=800),
    )

    if show_context:
        print("Retrieved context:\n")
        print(context)
        print("\nAnswer:\n")

    return response.text


In [ ]:
question = "Summerise Atharv Tembhurnikar's Resume in 4 points?"
print(answer_question(question, k=4, show_context=False))

Retrieved context:

[Context 1 | source=resume.pdf | page=1 | chunk=0]
Atharv Tembhurnikar
College Park, MD, USA
 +1 240-886-6670
# attem@umd.edu
ï LinkedIn
§ GitHub  Portfolio
Blogs IEEE Profile
Experience
Teaching Assistant (Agentic AI) – University of Maryland (UMD)
Aug 2026 – Present
Machine Learning Research Assistant – University of Maryland (UMD)
Dec 2025 – Jul 2026
• Developed ML-based pipelines for cell-wise annotation and intensity tracking on artificial-nose sensor data, enabling
structured capture of cellular responses and temporal patterns over time.
• Engineered interpretable feature representations (response amplitude, decay dynamics, temporal slopes) to quantify cell
behavior, supporting data-driven validation of experimental hypotheses and robust pattern classification.
Data Science Intern – Hackveda Private Limited (Pune, India)
Jan 2025 –

[Context 2 | source=resume.pdf | page=1 | chunk=5]
n 2024 - May 2024
∗Built an NLP-driven resume parsing system using spaCy + c

In [ ]:
sample_questions = [
    "Summarize my background in 4 bullet points.",
    "What projects should I highlight for an AI or backend role?",
    "What programming languages and tools are listed on my resume?",
    "What is a concise recruiter-style pitch about me?",
]

for q in sample_questions:
    print(f"\nQ: {q}\n")
    print(answer_question(q, k=4))

## Interactive loop

Run this cell when you want a tiny notebook chatbot. Type `exit` to stop.

In [ ]:
while True:
    user_question = input("Ask about the resume: ").strip()
    if user_question.lower() in {"exit", "quit", "q"}:
        break
    if not user_question:
        continue
    print("\n" + answer_question(user_question, k=4) + "\n")